# Notebook 3 — Train / Validation / Test Split

## Objective

The goal of this notebook is to split the labeled dataset into training, validation, and test sets while avoiding data leakage.

The split strategy will be selected after a small analysis of the dataset's time range and label distribution.

In [2]:
import pandas as pd

## 1. Load the Labeled Dataset

The labeled dataset created in Notebook 2 will be used as the input for this notebook.

In [3]:
labeled_table = pd.read_csv("../Artifacts/labeled_table.csv")

labeled_table.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,number_of_items,total_price,total_freight,number_of_payments,total_payment_value,delivery_label
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,29.99,8.72,3.0,38.71,on_time
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,118.70,22.76,1.0,141.46,on_time
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,159.90,19.22,1.0,179.12,on_time
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,45.00,27.20,1.0,72.20,on_time
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,19.90,8.72,1.0,28.62,on_time


## 2. Small Analysis Before Splitting

Before splitting the data, we will perform a small analysis to understand the time range of the orders and the distribution of the delivery labels. This will help us choose an appropriate splitting strategy.

In [4]:
labeled_table["order_purchase_timestamp"] = pd.to_datetime(
    labeled_table["order_purchase_timestamp"]
)

In [5]:
print(
    "Earliest order:",
    labeled_table["order_purchase_timestamp"].min()
)

print(
    "Latest order:",
    labeled_table["order_purchase_timestamp"].max()
)

Earliest order: 2016-09-15 12:16:38
Latest order: 2018-08-29 15:00:37


### Label Balance

We check the distribution of the target variable before splitting the data.

In [7]:
labeled_table["delivery_label"].value_counts(normalize=True).mul(100).round(2)

delivery_label
on_time    93.23
late        6.77
Name: proportion, dtype: float64

### Monthly Label Distribution

We examine the delivery label distribution across months to determine whether the target distribution changes over time.

In [8]:
monthly_label_balance = (
    labeled_table
    .assign(
        purchase_month=labeled_table["order_purchase_timestamp"].dt.to_period("M")
    )
    .groupby("purchase_month")["delivery_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("percentage")
    .reset_index()
)

monthly_label_balance

,purchase_month,delivery_label,percentage
0,2016-09,late,100.00
1,2016-10,on_time,99.26
2,2016-10,late,0.74
3,2016-12,on_time,100.00
4,2017-01,on_time,97.07
5,2017-01,late,2.93
6,2017-02,on_time,97.04
7,2017-02,late,2.96
8,2017-03,on_time,95.44
9,2017-03,late,4.56


## 3. Split Strategy

A time-based split was selected because the target distribution changes over time.

Using a time-based split allows the model to learn from historical orders and evaluate its performance on more recent orders, which better reflects a real-world prediction scenario.

The data will be split chronologically into:

* **70% Train**
* **15% Validation**
* **15% Test**

The test set will represent the most recent period and will remain isolated until final model evaluation.

In [9]:
labeled_table_sorted = (
    labeled_table
    .sort_values("order_purchase_timestamp")
    .reset_index(drop=True)
)

In [10]:
n = len(labeled_table_sorted)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

print("Total rows:", n)
print("Train end:", train_end)
print("Validation end:", validation_end)

Total rows: 96476
Train end: 67533
Validation end: 82004


In [11]:
train = labeled_table_sorted.iloc[:train_end].copy()

validation = labeled_table_sorted.iloc[train_end:validation_end].copy()

test = labeled_table_sorted.iloc[validation_end:].copy()

## 4. Verify the Splits

We verify the number of rows and the date range of each split to ensure that the data was divided chronologically.

In [12]:
for name, dataset in {
    "Train": train,
    "Validation": validation,
    "Test": test
}.items():
    
    print(f"\n{name}")
    print("Rows:", len(dataset))
    print("Start date:", dataset["order_purchase_timestamp"].min())
    print("End date:", dataset["order_purchase_timestamp"].max())


Train
Rows: 67533
Start date: 2016-09-15 12:16:38
End date: 2018-04-15 20:07:56

Validation
Rows: 14471
Start date: 2018-04-15 20:10:23
End date: 2018-06-21 07:50:39

Test
Rows: 14472
Start date: 2018-06-21 08:29:29
End date: 2018-08-29 15:00:37


### Label Balance Across Splits

We compare the target label distribution across the training, validation, and test sets.

In [13]:
for name, dataset in {
    "Train": train,
    "Validation": validation,
    "Test": test
}.items():
    
    print(f"\n{name}")
    print(
        dataset["delivery_label"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )


Train
delivery_label
on_time    92.17
late        7.83
Name: proportion, dtype: float64

Validation
delivery_label
on_time    95.69
late        4.31
Name: proportion, dtype: float64

Test
delivery_label
on_time    95.72
late        4.28
Name: proportion, dtype: float64


### Check for Order Overlap

We verify that no order appears in more than one split.

In [14]:
train_ids = set(train["order_id"])
validation_ids = set(validation["order_id"])
test_ids = set(test["order_id"])

print("Train & Validation overlap:", len(train_ids & validation_ids))
print("Train & Test overlap:", len(train_ids & test_ids))
print("Validation & Test overlap:", len(validation_ids & test_ids))

Train & Validation overlap: 0
Train & Test overlap: 0
Validation & Test overlap: 0


### Check Total Rows

We verify that all rows from the original dataset are preserved across the three splits.

In [15]:
print("Original rows:", len(labeled_table_sorted))
print("Total split rows:", len(train) + len(validation) + len(test))

Original rows: 96476
Total split rows: 96476


## 5. Save Split Artifacts

The final train, validation, and test datasets are saved as CSV files for use in the following notebooks.

In [16]:
train.to_csv("../Artifacts/train.csv", index=False)
validation.to_csv("../Artifacts/validation.csv", index=False)
test.to_csv("../Artifacts/test.csv", index=False)